# Cypher span grounding — one-sample forward walkthrough

Notebook này minh họa riêng nhánh **Cypher span grounding** của CypherKD:

`gold Cypher → character spans → token spans → span representations → question/schema context → relation loss`

Notebook không tạo optimizer, không gọi `backward()` và không cập nhật trọng số. Các cell trước phần model forward vẫn chạy được khi không có GPU.

## 0. Cấu hình

- Để chỉ demo extraction/alignment, giữ `RUN_MODEL_FORWARD = False`.
- Để tính span-context relation loss thật, đổi thành `True`. Student 0.6B + teacher 4B thường cần GPU khoảng 12–16 GiB VRAM.
- Có thể truyền adapter đã train sẵn; để `None` nếu chỉ muốn minh họa forward của base models.

In [ ]:
from pathlib import Path
import sys

root_candidates = [
    Path.cwd(),
    Path.cwd() / "CypherKD-draft",  # Kaggle: clone repo vào /kaggle/working
    Path.cwd() / "CypherKD",
    Path.cwd().parent,
]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "benchmarks").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Không tìm thấy repo. Hãy clone CypherKD vào thư mục làm việc.")
sys.path.insert(0, str(PROJECT_ROOT))

DATA_FILE = PROJECT_ROOT / "benchmarks/Cypherbench/train.jsonl"
# Training sample có question/schema alignment rõ và nhiều loại Cypher span.
SAMPLE_INDEX = 108
# Các sample khác có khả năng thể hiện question attention rõ hơn:
# SAMPLE_INDEX = 251   # Art: genre + displayedAt + Museum + creation_year + sorting.
# SAMPLE_INDEX = 6226  # Biology: schema ngắn; hasParent + hasRank + longest_lifespan_years.
# SAMPLE_INDEX = 4369  # Biology: hasParent + hasRank + aggregation count.
# SAMPLE_INDEX = 868   # Art: question dài; hasGenre + Painting/Sculpture + creation_year.
# SAMPLE_INDEX = 6090  # Art: shared displayedAt relation + creation_year ordering.
MAX_LENGTH = 1024

RUN_MODEL_FORWARD = False
STUDENT_MODEL = "Qwen/Qwen3-0.6B"
TEACHER_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
STUDENT_ADAPTER = None
TEACHER_ADAPTER = None
STUDENT_LAYER = -1
TEACHER_LAYER = -1

print(f"Project root : {PROJECT_ROOT}")
print(f"Data file    : {DATA_FILE}")
print(f"Model forward: {RUN_MODEL_FORWARD}")

## 1. Đọc đúng một training sample

In [ ]:
import json
from IPython.display import display, Markdown

def read_jsonl_item(path: Path, index: int):
    if index < 0:
        raise ValueError("SAMPLE_INDEX phải không âm")
    with path.open("r", encoding="utf-8") as handle:
        for current, line in enumerate(handle):
            if current == index:
                return json.loads(line)
    raise IndexError(f"Không có sample {index} trong {path}")

sample = read_jsonl_item(DATA_FILE, SAMPLE_INDEX)
system_prompt = sample["system_prompt"]
user_prompt = sample["user_prompt"]
raw_response_text = sample["response"]
response_json = json.loads(raw_response_text)
# Chuẩn hóa escape Unicode (ví dụ \\u00e9 → é) để parser có thể định vị
# nguyên văn Cypher trong response bằng character offsets.
response_text = json.dumps(response_json, ensure_ascii=False)
gold_cypher = response_json["cypher"]

question = user_prompt.split("QUESTION:\n", 1)[-1].split("\n\nSCHEMA:\n", 1)[0]
schema_text = user_prompt.split("\n\nSCHEMA:\n", 1)[-1].split(
    "\n\nGenerate a Cypher query", 1
)[0]
display(Markdown(f"**Question**\n\n> {question}"))
display(Markdown(f"**Schema**\n\n```json\n{schema_text}\n```"))
display(Markdown(f"**Gold Cypher**\n\n```cypher\n{gold_cypher}\n```"))

## 2. Trích xuất Cypher spans

Dùng chính parser của dataset hiện tại. Bốn loại span được trích xuất là `clause`, `triplet`, `node_pattern` và `expression`.

In [ ]:
import html
from data_utils.lm_datasets import extract_text2cypher_span_items_from_response

span_items = extract_text2cypher_span_items_from_response(response_text)
if not span_items:
    raise RuntimeError("Không extract được Cypher span từ response JSON")

print(f"Extracted {len(span_items)} spans\n")
print(f"{'#':>2}  {'type':<14} {'start':>5} {'end':>5}  text")
print("-" * 88)
for idx, item in enumerate(span_items):
    print(f"{idx:>2}  {item['type']:<14} {item['start']:>5} {item['end']:>5}  {item['text']}")

SPAN_COLORS = {
    "clause": "#fde68a",
    "triplet": "#bfdbfe",
    "node_pattern": "#bbf7d0",
    "expression": "#fecaca",
}

def highlight_one_type(text, items, span_type):
    selected = sorted(
        (item for item in items if item["type"] == span_type),
        key=lambda item: item["start"],
    )
    cursor, chunks = 0, []
    for item in selected:
        chunks.append(html.escape(text[cursor:item["start"]]))
        chunks.append(
            f'<mark style="background:{SPAN_COLORS[span_type]}; padding:2px 3px;">'
            f'{html.escape(text[item["start"]:item["end"]])}</mark>'
        )
        cursor = item["end"]
    chunks.append(html.escape(text[cursor:]))
    return "".join(chunks)

for span_type in SPAN_COLORS:
    display(Markdown(f"**{span_type}**"))
    display({"text/html": f'<code style="white-space:pre-wrap">{highlight_one_type(response_text, span_items, span_type)}</code>'}, raw=True)

## 3. Tokenize prompt + gold response

`labels == -100` đánh dấu prompt tokens không tham gia supervised loss. Trong notebook này labels chỉ được dùng để xác định ranh giới prompt/response và context mask; không tính LM loss.

In [ ]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, padding_side="right", trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]
try:
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
except TypeError:
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

full_text = prompt_text + response_text
encoded = tokenizer(
    full_text,
    add_special_tokens=False,
    return_offsets_mapping=True,
    return_tensors="pt",
)
full_ids = encoded["input_ids"][0].tolist() + [tokenizer.eos_token_id]
prompt_length = len(tokenizer.encode(prompt_text, add_special_tokens=False))
if len(full_ids) > MAX_LENGTH:
    raise RuntimeError(
        f"Sample có {len(full_ids)} tokens > MAX_LENGTH={MAX_LENGTH}. "
        "Hãy tăng MAX_LENGTH hoặc chọn sample khác để không cắt mất span."
    )

# Cùng convention với LMTrainDataset: input token t dự đoán label token t+1.
input_ids = torch.tensor([full_ids[:-1]], dtype=torch.long)
attention_mask = torch.ones_like(input_ids)
labels = torch.tensor([full_ids[1:]], dtype=torch.long)
labels[:, : max(0, prompt_length - 1)] = -100

offsets = encoded["offset_mapping"][:, : input_ids.shape[1], :]
response_start = full_text.find(response_text)
absolute_span_offsets = [
    (response_start + item["start"], response_start + item["end"])
    for item in span_items
]

print(f"Prompt tokens   : {prompt_length}")
print(f"Response tokens : {(labels != -100).sum().item()}")
print(f"Model input     : {tuple(input_ids.shape)}")
print(f"Offset mapping  : {tuple(offsets.shape)}")

## 4. Character span → token span

In [ ]:
def build_token_to_span_map(attention_mask, offsets_mapping, span_offsets):
    batch_size, seq_len = attention_mask.shape
    max_spans = max((len(items) for items in span_offsets), default=0)
    span_starts = torch.zeros(batch_size, max_spans, dtype=torch.long)
    span_ends = torch.zeros(batch_size, max_spans, dtype=torch.long)
    span_mask = torch.zeros(batch_size, max_spans, dtype=torch.bool)
    for batch_idx, items in enumerate(span_offsets):
        if items:
            values = torch.tensor(items, dtype=torch.long)
            span_starts[batch_idx, :len(items)] = values[:, 0]
            span_ends[batch_idx, :len(items)] = values[:, 1]
            span_mask[batch_idx, :len(items)] = True
    token_start = offsets_mapping[..., 0].unsqueeze(-1)
    token_end = offsets_mapping[..., 1].unsqueeze(-1)
    token_in_span = (
        (token_start + 1 >= span_starts.unsqueeze(1))
        & (token_end <= span_ends.unsqueeze(1))
        & attention_mask.unsqueeze(-1).bool()
        & span_mask.unsqueeze(1)
    )
    return token_in_span, span_mask

token_to_span_map, span_mask = build_token_to_span_map(
    attention_mask, offsets, [absolute_span_offsets]
)

for span_idx, item in enumerate(span_items):
    token_indices = torch.nonzero(token_to_span_map[0, :, span_idx], as_tuple=False).flatten().tolist()
    token_ids = input_ids[0, token_indices].tolist()
    token_text = tokenizer.convert_ids_to_tokens(token_ids)
    print(f"[{span_idx:02d}] {item['type']:<14} {item['text']}")
    print(f"     token indices: {token_indices}")
    print(f"     tokens       : {token_text}\n")

assert span_mask.all(), "Có span không hợp lệ"
assert token_to_span_map.any(dim=1).all(), "Có character span không map được sang token"

## 5. Question/schema context masks

Logic dưới đây dùng cùng các marker `QUESTION`, `SCHEMA` và phần kết prompt như training code.

In [ ]:
QUESTION_MARKER = "QUESTION:\n"
SCHEMA_MARKER = "SCHEMA:\n"
SCHEMA_END_MARKER = (
    "\n\nGenerate a Cypher query that answers the question using only the provided schema.\n"
    "Return only the JSON object in the required format."
)

def find_subsequence(sequence, pattern, start=0):
    for idx in range(start, len(sequence) - len(pattern) + 1):
        if sequence[idx:idx + len(pattern)] == pattern:
            return idx, len(pattern)
    return -1, 0

def build_question_schema_masks(tokenizer, input_ids, attention_mask, labels):
    prompt_mask = (labels == -100) & attention_mask.bool()
    question_mask = torch.zeros_like(prompt_mask)
    schema_mask = torch.zeros_like(prompt_mask)
    q_marker = tokenizer.encode(QUESTION_MARKER, add_special_tokens=False)
    s_marker = tokenizer.encode(SCHEMA_MARKER, add_special_tokens=False)
    end_marker = tokenizer.encode(SCHEMA_END_MARKER, add_special_tokens=False)

    for batch_idx in range(input_ids.size(0)):
        prompt_indices = torch.nonzero(prompt_mask[batch_idx], as_tuple=False).flatten()
        ids = input_ids[batch_idx, prompt_indices].tolist()
        q_pos, q_len = find_subsequence(ids, q_marker)
        s_pos, s_len = find_subsequence(ids, s_marker, q_pos + q_len)
        end_pos, _ = find_subsequence(ids, end_marker, s_pos + s_len)
        if q_pos < 0 or s_pos < 0:
            continue
        if end_pos < 0:
            end_pos = len(ids)
        question_mask[batch_idx, prompt_indices[q_pos + q_len:s_pos]] = True
        schema_mask[batch_idx, prompt_indices[s_pos + s_len:end_pos]] = True
    return question_mask, schema_mask

question_mask, schema_mask = build_question_schema_masks(
    tokenizer, input_ids, attention_mask, labels
)
source_mask = question_mask | schema_mask

def decode_mask(mask):
    ids = input_ids[0, mask[0]].tolist()
    return tokenizer.decode(ids, skip_special_tokens=True)

print(f"Question-mask tokens: {question_mask.sum().item()}")
print(f"Schema-mask tokens  : {schema_mask.sum().item()}")
print(f"Question preview    : {decode_mask(question_mask)[:250]!r}")
print(f"Schema preview      : {decode_mask(schema_mask)[:250]!r}")

assert question_mask.any(), "Question mask rỗng"
assert schema_mask.any(), "Schema mask rỗng"

## 6. Tùy chọn: student/teacher forward

Cell này chỉ chạy khi `RUN_MODEL_FORWARD=True`. Cả hai model ở `eval()` và toàn bộ forward nằm trong `torch.inference_mode()`.

In [ ]:
from transformers import AutoModelForCausalLM

def load_model(model_name, adapter_path, device, dtype):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map={"": str(device)},
        trust_remote_code=True,
    )
    if adapter_path:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_path)
        model = model.merge_and_unload()
    model.eval()
    return model

student_selected_hidden = teacher_selected_hidden = None
student_probe_before = student_probe_after = None

if RUN_MODEL_FORWARD:
    if not torch.cuda.is_available():
        raise RuntimeError("Model-forward demo cần CUDA; extraction cells phía trên không cần GPU.")
    device = torch.device("cuda")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    student = load_model(STUDENT_MODEL, STUDENT_ADAPTER, device, dtype)
    teacher = load_model(TEACHER_MODEL, TEACHER_ADAPTER, device, dtype)
    model_inputs = {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device),
    }
    first_student_parameter = next(student.parameters())
    student_probe_before = first_student_parameter.detach().flatten()[:1024].cpu().clone()
    with torch.inference_mode():
        student_result = student(**model_inputs, output_hidden_states=True, use_cache=False)
        student_selected_hidden = student_result.hidden_states[STUDENT_LAYER]
        del student_result
        teacher_result = teacher(**model_inputs, output_hidden_states=True, use_cache=False)
        teacher_selected_hidden = teacher_result.hidden_states[TEACHER_LAYER]
        del teacher_result
    student_probe_after = first_student_parameter.detach().flatten()[:1024].cpu().clone()
    print(f"Student selected hidden: {tuple(student_selected_hidden.shape)}")
    print(f"Teacher selected hidden: {tuple(teacher_selected_hidden.shape)}")
else:
    print("SKIPPED. Đổi RUN_MODEL_FORWARD=True rồi chạy lại từ cell cấu hình để tính relation loss.")

## 7. Compute intermediate tensors

Teacher tạo attention từ từng Cypher span về question/schema. Student dùng cùng attention weights để hai quan hệ có thể so sánh trực tiếp.

In [ ]:
import math
import torch.nn.functional as F

EPS = 1e-5
NEG_INF = -1e4

def compute_token_weights(hidden_state, attention_mask):
    hidden = hidden_state.float()
    mask = attention_mask.bool()
    std = hidden.std(dim=-1, keepdim=True, unbiased=False).clamp(min=EPS)
    normalized = hidden / std
    scores = torch.matmul(normalized, normalized.transpose(1, 2)) / math.sqrt(hidden.size(-1))
    pair_mask = mask.unsqueeze(1) & mask.unsqueeze(2)
    scores = scores.masked_fill(~pair_mask, NEG_INF)
    diagonal = torch.eye(scores.size(-1), device=scores.device, dtype=torch.bool).unsqueeze(0)
    scores = scores.masked_fill(diagonal, NEG_INF)
    probabilities = torch.softmax(scores, dim=-1)
    probabilities = torch.nan_to_num(probabilities) * pair_mask.float()
    probabilities = probabilities / probabilities.sum(dim=-1, keepdim=True).clamp(min=EPS)
    weights = probabilities.sum(dim=1) / mask.float().sum(dim=1, keepdim=True).clamp(min=1.0)
    return weights * mask.float()

def compute_span_representations(hidden_state, token_map, span_mask, token_weights):
    weighted_map = token_map.float() * token_weights.float().unsqueeze(-1)
    span_sum = torch.einsum("bld,bls->bsd", hidden_state.float(), weighted_map)
    denominator = weighted_map.sum(dim=1).unsqueeze(-1).clamp(min=EPS)
    return (span_sum / denominator) * span_mask.unsqueeze(-1).float()

def compute_context(hidden_state, span_repr, span_mask, source_mask, shared_attention=None):
    hidden = hidden_state.float()
    relevance_scores = None
    if shared_attention is None:
        scores = torch.matmul(span_repr.float(), hidden.transpose(1, 2)) / math.sqrt(hidden.size(-1))
        relevance_scores = scores.masked_fill(~source_mask.unsqueeze(1), float("nan"))
        scores = scores.masked_fill(~source_mask.unsqueeze(1), NEG_INF)
        attention = torch.softmax(scores, dim=-1)
        attention = torch.nan_to_num(attention) * source_mask.unsqueeze(1).float()
        attention = attention / attention.sum(dim=-1, keepdim=True).clamp(min=EPS)
    else:
        attention = shared_attention.float()
    attention = attention * span_mask.unsqueeze(-1).float()
    context = torch.matmul(attention, hidden) * span_mask.unsqueeze(-1).float()
    return context, attention, relevance_scores

span_forward = None
if RUN_MODEL_FORWARD:
    device = student_selected_hidden.device
    token_map_gpu = token_to_span_map.to(device)
    span_mask_gpu = span_mask.to(device)
    attention_gpu = attention_mask.to(device)
    source_gpu = source_mask.to(device)
    question_gpu = question_mask.to(device)
    schema_gpu = schema_mask.to(device)
    student_hidden = student_selected_hidden
    teacher_hidden = teacher_selected_hidden

    student_token_weights = compute_token_weights(student_hidden, attention_gpu)
    teacher_token_weights = compute_token_weights(teacher_hidden, attention_gpu)
    raw_span_weights = (token_map_gpu.float() * teacher_token_weights.unsqueeze(-1)).sum(dim=1)
    raw_span_weights = raw_span_weights * span_mask_gpu.float()
    normalized_span_weights = raw_span_weights / raw_span_weights.sum(dim=-1, keepdim=True).clamp(min=EPS)

    student_span = compute_span_representations(
        student_hidden, token_map_gpu, span_mask_gpu, student_token_weights
    )
    teacher_span = compute_span_representations(
        teacher_hidden, token_map_gpu, span_mask_gpu, teacher_token_weights
    )
    teacher_context, teacher_attention, teacher_relevance = compute_context(
        teacher_hidden, teacher_span, span_mask_gpu, source_gpu
    )
    student_context, _, _ = compute_context(
        student_hidden, student_span, span_mask_gpu, source_gpu,
        shared_attention=teacher_attention,
    )

    student_similarity = F.cosine_similarity(student_span, student_context, dim=-1)
    teacher_similarity = F.cosine_similarity(teacher_span, teacher_context, dim=-1)
    per_span_loss = (student_similarity - teacher_similarity).pow(2)
    span_contributions = normalized_span_weights * per_span_loss
    relation_loss = (
        (per_span_loss * normalized_span_weights).sum()
        / normalized_span_weights.sum().clamp(min=EPS)
    )
    question_attention = (teacher_attention * question_gpu.unsqueeze(1)).sum(dim=-1)
    schema_attention = (teacher_attention * schema_gpu.unsqueeze(1)).sum(dim=-1)
    question_token_count = question_gpu.sum(dim=-1, keepdim=True).clamp(min=1)
    schema_token_count = schema_gpu.sum(dim=-1, keepdim=True).clamp(min=1)
    question_density = question_attention / question_token_count
    schema_density = schema_attention / schema_token_count
    question_density_share = question_density / (
        question_density + schema_density
    ).clamp(min=EPS)

    span_forward = {
        "student_hidden_shape": tuple(student_hidden.shape),
        "teacher_hidden_shape": tuple(teacher_hidden.shape),
        "student_token_weights": student_token_weights.detach().cpu(),
        "teacher_token_weights": teacher_token_weights.detach().cpu(),
        "student_span": student_span.detach().cpu(),
        "teacher_span": teacher_span.detach().cpu(),
        "teacher_relevance": teacher_relevance.detach().cpu(),
        "teacher_attention": teacher_attention.detach().cpu(),
        "student_context": student_context.detach().cpu(),
        "teacher_context": teacher_context.detach().cpu(),
        "student_similarity": student_similarity.detach().cpu(),
        "teacher_similarity": teacher_similarity.detach().cpu(),
        "per_span_loss": per_span_loss.detach().cpu(),
        "raw_span_weights": raw_span_weights.detach().cpu(),
        "span_weights": normalized_span_weights.detach().cpu(),
        "span_contributions": span_contributions.detach().cpu(),
        "question_attention": question_attention.detach().cpu(),
        "schema_attention": schema_attention.detach().cpu(),
        "question_density": question_density.detach().cpu(),
        "schema_density": schema_density.detach().cpu(),
        "question_density_share": question_density_share.detach().cpu(),
        "relation_loss": relation_loss.detach().cpu(),
    }
else:
    print("SKIPPED because RUN_MODEL_FORWARD=False")

## 8. Hidden states, token importance và span representations

Với model $m\in\{T,S\}$, hidden states tại layer được chọn là $H^{(m)}\in\mathbb{R}^{L\times d_m}$. Token importance trong implementation được tính bởi:

$$\widehat H_t^{(m)}=\frac{H_t^{(m)}}{\sigma(H_t^{(m)})}$$

$$a_{s\rightarrow t}^{(m)}=\operatorname{softmax}_{t}\left(\frac{\widehat H_s^{(m)\top}\widehat H_t^{(m)}}{\sqrt{d_m}}\right),\qquad w_t^{(m)}=\frac{1}{N}\sum_s a_{s\rightarrow t}^{(m)}$$

Sau đó biểu diễn span $S_k$ là weighted pooling:

$$U_k^{(m)}=\frac{\sum_{t\in S_k}w_t^{(m)}H_t^{(m)}}{\sum_{t\in S_k}w_t^{(m)}}$$

In [ ]:
if span_forward is not None:
    print(f"Student H shape: {span_forward['student_hidden_shape']}")
    print(f"Teacher H shape: {span_forward['teacher_hidden_shape']}")

    cypher_token_mask = token_to_span_map[0].any(dim=-1)
    cypher_token_indices = torch.nonzero(cypher_token_mask, as_tuple=False).flatten().tolist()
    print("\nToken importance for tokens covered by at least one Cypher span")
    print(f"{'idx':>5} {'token':<24} {'w_student':>12} {'w_teacher':>12}")
    print("-" * 60)
    for token_idx in cypher_token_indices:
        token_text = tokenizer.convert_ids_to_tokens(int(input_ids[0, token_idx]))
        print(
            f"{token_idx:>5} {token_text[:24]:<24} "
            f"{span_forward['student_token_weights'][0, token_idx].item():>12.7f} "
            f"{span_forward['teacher_token_weights'][0, token_idx].item():>12.7f}"
        )

    print("\nSpan representations (vector preview shows first 4 dimensions)")
    for span_idx, item in enumerate(span_items):
        u_s = span_forward['student_span'][0, span_idx]
        u_t = span_forward['teacher_span'][0, span_idx]
        print(f"[{span_idx:02d}] {item['type']}: {item['text']}")
        print(f"     U^S shape={tuple(u_s.shape)}, norm={u_s.norm().item():.4f}, preview={u_s[:4].tolist()}")
        print(f"     U^T shape={tuple(u_t.shape)}, norm={u_t.norm().item():.4f}, preview={u_t[:4].tolist()}")
else:
    print("SKIPPED because RUN_MODEL_FORWARD=False")

## 9. Teacher-guided question-schema pooling

Chỉ question và schema tokens thuộc context set:

$$\mathcal C=\{t:t\in QUESTION\ \lor\ t\in SCHEMA\}$$

Teacher tính relevance và attention cho từng span:

$$e_{k,t}^{(T)}=\frac{U_k^{(T)\top}H_t^{(T)}}{\sqrt{d_T}},\qquad \alpha_{k,t}^{(T)}=\operatorname{softmax}_{t\in\mathcal C}(e_{k,t}^{(T)})$$

Cùng teacher attention được dùng để pool context cho cả hai model:

$$q_k^{(T)}=\sum_{t\in\mathcal C}\alpha_{k,t}^{(T)}H_t^{(T)},\qquad q_k^{(S)}=\sum_{t\in\mathcal C}\alpha_{k,t}^{(T)}H_t^{(S)}$$

In [ ]:
if span_forward is not None:
    source_indices = torch.nonzero(source_mask[0], as_tuple=False).flatten()
    for span_idx, item in enumerate(span_items):
        alpha = span_forward['teacher_attention'][0, span_idx]
        relevance = span_forward['teacher_relevance'][0, span_idx]
        source_alpha = alpha[source_indices]
        top_count = min(5, source_indices.numel())
        top_values, top_positions = torch.topk(source_alpha, k=top_count)
        q_s = span_forward['student_context'][0, span_idx]
        q_t = span_forward['teacher_context'][0, span_idx]

        print(f"\n[{span_idx:02d}] {item['type']}: {item['text']}")
        print(f"  sum(alpha over C) = {alpha[source_mask[0]].sum().item():.6f}")
        print(f"  q^S shape={tuple(q_s.shape)}, norm={q_s.norm().item():.4f}, preview={q_s[:4].tolist()}")
        print(f"  q^T shape={tuple(q_t.shape)}, norm={q_t.norm().item():.4f}, preview={q_t[:4].tolist()}")
        print(f"  {'source':<8} {'token':<28} {'e_teacher':>12} {'alpha':>12}")
        for alpha_value, local_pos in zip(top_values.tolist(), top_positions.tolist()):
            token_idx = int(source_indices[local_pos])
            region = "QUESTION" if bool(question_mask[0, token_idx]) else "SCHEMA"
            token_text = tokenizer.decode([int(input_ids[0, token_idx])], skip_special_tokens=False)
            print(
                f"  {region:<8} {token_text.strip()[:28]:<28} "
                f"{relevance[token_idx].item():>12.5f} {alpha_value:>12.7f}"
            )

    valid_alpha_sums = span_forward['teacher_attention'][0, span_mask[0]][:, source_mask[0]].sum(dim=-1)
    assert torch.allclose(valid_alpha_sums, torch.ones_like(valid_alpha_sums), atol=1e-5)
else:
    print("SKIPPED because RUN_MODEL_FORWARD=False")

## 10. Relation vectors, span weights và $\mathcal L_{rel}$

Mỗi span tạo một relation score và toàn bộ scores tạo thành relation vector:

$$r_k^{(m)}=\cos(U_k^{(m)},q_k^{(m)}),\quad m\in\{T,S\}$$

$$\mathbf r^{(T)}=[r_1^{(T)},\ldots,r_{N_{sp}}^{(T)}],\qquad \mathbf r^{(S)}=[r_1^{(S)},\ldots,r_{N_{sp}}^{(S)}]$$

Teacher-derived raw và normalized span importance:

$$\widetilde\omega_k^{sp}=\sum_{t\in S_k}w_t^{(T)},\qquad \omega_k^{sp}=\frac{\widetilde\omega_k^{sp}}{\sum_{j=1}^{N_{sp}}\widetilde\omega_j^{sp}}$$

Relation loss và contribution của từng span:

$$c_k=\omega_k^{sp}(r_k^{(S)}-r_k^{(T)})^2,\qquad \mathcal L_{rel}=\sum_{k=1}^{N_{sp}}c_k$$

Implementation chia thêm cho $\sum_k\omega_k^{sp}$ để an toàn số học; vì weights đã chuẩn hóa nên hai biểu thức tương đương.

`Q attn` và `Schema attn` là tổng attention mass. Vì schema thường dài hơn question nhiều lần, notebook còn báo cáo attention density trên mỗi token:

$$d_Q = \frac{Q\_attn}{N_Q}, \qquad d_S = \frac{Schema\_attn}{N_S}$$

`Q share` là density đã chuẩn hóa: $d_Q/(d_Q+d_S)$. Giá trị gần `0.5` nghĩa là một question token và một schema token nhận attention trung bình tương đương; giá trị lớn hơn `0.5` nghĩa là attention ưu tiên question sau khi đã loại ảnh hưởng độ dài.

In [ ]:
if span_forward is not None:
    valid_spans = span_mask[0]
    relation_vector_s = span_forward['student_similarity'][0, valid_spans]
    relation_vector_t = span_forward['teacher_similarity'][0, valid_spans]
    print(f"Student relation vector r^S ({relation_vector_s.numel()} dims):")
    print(relation_vector_s.tolist())
    print(f"\nTeacher relation vector r^T ({relation_vector_t.numel()} dims):")
    print(relation_vector_t.tolist())
    print()
    print(
        f"{'#':>2} {'type':<13} {'span':<34} "
        f"{'r^S':>9} {'r^T':>9} {'diff²':>9} {'raw ω':>9} {'ω':>9} {'contrib':>10} "
        f"{'Q attn':>9} {'Schema attn':>11} {'Q density':>10} {'Schema dens':>11} {'Q share':>9}"
    )
    print("-" * 198)
    for idx, item in enumerate(span_items):
        text = item["text"].replace("\n", " ")
        text = text if len(text) <= 34 else text[:31] + "..."
        print(
            f"{idx:>2} {item['type']:<13} {text:<34} "
            f"{span_forward['student_similarity'][0, idx].item():>9.4f} "
            f"{span_forward['teacher_similarity'][0, idx].item():>9.4f} "
            f"{span_forward['per_span_loss'][0, idx].item():>9.4f} "
            f"{span_forward['raw_span_weights'][0, idx].item():>9.6f} "
            f"{span_forward['span_weights'][0, idx].item():>9.4f} "
            f"{span_forward['span_contributions'][0, idx].item():>10.6f} "
            f"{span_forward['question_attention'][0, idx].item():>9.4f} "
            f"{span_forward['schema_attention'][0, idx].item():>11.4f} "
            f"{span_forward['question_density'][0, idx].item():>10.6f} "
            f"{span_forward['schema_density'][0, idx].item():>11.6f} "
            f"{span_forward['question_density_share'][0, idx].item():>9.4f}"
        )

    relation_value = span_forward["relation_loss"].item()
    valid_q_share = span_forward["question_density_share"][0, span_mask[0]]
    omega_sum = span_forward['span_weights'][0, valid_spans].sum()
    contribution_sum = span_forward['span_contributions'][0, valid_spans].sum()
    weights_unchanged = torch.equal(student_probe_before, student_probe_after)
    print("\nSummary")
    print(f"  Sum normalized span weights : {omega_sum.item():.6f}")
    print(f"  Sum span contributions      : {contribution_sum.item():.6f}")
    print(f"  L_rel                       : {relation_value:.6f}")
    print(f"  Mean Q density share   : {valid_q_share.mean().item():.4f}")
    print(f"  Finite loss            : {torch.isfinite(span_forward['relation_loss']).item()}")
    print(f"  Optimizer created      : NO")
    print(f"  Backward called        : NO")
    print(f"  Student weights changed: {'NO' if weights_unchanged else 'YES — unexpected'}")

    assert torch.isfinite(span_forward["relation_loss"]), "Relation loss không hữu hạn"
    assert torch.all((relation_vector_s >= -1) & (relation_vector_s <= 1))
    assert torch.all((relation_vector_t >= -1) & (relation_vector_t <= 1))
    assert torch.allclose(omega_sum, torch.tensor(1.0), atol=1e-5)
    assert torch.allclose(contribution_sum, span_forward['relation_loss'], atol=1e-6)
    assert weights_unchanged, "Student weights đã thay đổi ngoài dự kiến"
else:
    print("Extraction/alignment walkthrough completed.")
    print("Bật RUN_MODEL_FORWARD=True để hiện bảng student/teacher và relation loss.")